# 01 — Macro History and Satellite Model

**Wearing the Model Development hat.** Builds the historical macro/loss dataset and fits the satellite model: a linear regression of portfolio loss rate on lagged unemployment, GDP growth, and HPI growth.

In [ ]:
import sys
sys.path.append('../src')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
df = pd.read_csv('../data/raw/macro_and_loss_history.csv')
print(df.shape)
df.head()

## Historical macro and loss trajectories

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axes[0].plot(df['unemployment_rate'], label='Unemployment Rate (%)', color='#1F3864')
axes[0].plot(df['hpi_growth'], label='HPI Growth (%)', color='#C00000')
axes[0].legend(); axes[0].set_title('Macro History')
axes[1].plot(df['portfolio_loss_rate'], color='#548235', label='Portfolio Loss Rate (%)')
axes[1].legend(); axes[1].set_title('Portfolio Loss Rate')
plt.tight_layout()
plt.show()

Note the lagged relationship: the loss rate peak occurs roughly one quarter after the unemployment peak, consistent with the lag structure built into the satellite model.

## Fit the satellite model

In [ ]:
from satellite_model import fit_satellite_model, model_summary
model, prepared = fit_satellite_model(df)
summary = model_summary(model, prepared)
print(f"R-squared: {summary['r_squared']:.4f}")
print(f"Intercept: {summary['intercept']:.4f}")
for feat, coef in summary['coefficients'].items():
    print(f'  {feat:28s}: {coef:+.4f}')

## Conceptual soundness check: correlation among explanatory variables

Before trusting these coefficients individually, check whether the explanatory variables are themselves highly correlated -- a classic validation step that's easy to skip if you only look at overall R².

In [ ]:
prepared[['unemployment_rate_lag1', 'gdp_growth', 'hpi_growth']].corr()

**Finding:** unemployment and HPI growth are correlated at roughly -0.95 in this sample -- severe multicollinearity. This is investigated further in notebook 03 and is the basis for Finding 1 in the validation report. It also explains the mildly counterintuitive positive sign on the GDP growth coefficient above.

## Training data ranges

These ranges are the benchmark against which CCAR scenario inputs will be checked for extrapolation in the next notebook.

In [ ]:
for feat, (lo, hi) in summary['training_ranges'].items():
    print(f'{feat:28s}: [{lo:.2f}, {hi:.2f}]')